In [ ]:
%load_ext autoreload
%autoreload 2

# Imports, setup, data loading

## Imports

In [ ]:
# Imports required packages and functions
import os
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from typing import Optional, Sequence
from autogluon.tabular import TabularPredictor
from make_clinical_dataset.epr.combine import merge_closest_measurements
from make_clinical_dataset.shared.constants import ROOT_DIR
from ml_common.summary import get_label_distribution
from ml_common.autogluon import train_models, evaluate
from ml_common.eval import auc_scores

## Setup

In [ ]:
# Show more columns in DataFrame output
pd.set_option('display.max_columns', 100)
pd.set_option("display.max_rows", None)

## Read datasets

In [ ]:
# Path to directories
DATE = '2025-03-29'
DATA_DIR = f"{ROOT_DIR}/data/final/data_{DATE}"
INFO_DIR = f"/cluster/projects/gliugroup/2BLAST/data/info"
WORKING_DIR = f"/cluster/projects/gliugroup/work_dir/elnaz_ziad/go-treat"

In [ ]:
# Load EHR data
main = pd.read_parquet(f'{DATA_DIR}/processed/treatment_centered_data.parquet')
dates = pd.read_parquet(f'{DATA_DIR}/processed/treatment_centered_dates.parquet')

# Load OACC data
carg = pd.read_csv('/cluster/home/t128190uhn/datasets/oacc/cleaned/cleaned_and_filtered_oacc_data.csv', parse_dates=['date_referred'])

carg = carg[['mrn', 'date_referred', 'carg_toxicity_risk']]

# Load CT data
ct = pd.read_parquet("/cluster/home/t128190uhn/datasets/clinical_trials/cleaning/ct.parquet")

# Load lookup tables
religion_names_normalized = pd.read_csv(f'{INFO_DIR}/religion_names_normalized.csv')
language_names_normalized = pd.read_csv(f'{INFO_DIR}/language_names_normalized.csv')

In [ ]:
# Record row and unique MRN counts for patient flow reporting
n_rows_all = len(main)
n_mrns_all = main['mrn'].nunique()

# Define functions locally
Temporary for testing; functions will be moved to the appropriate repo (e.g., ml-commons) once finalized.

In [ ]:
def create_composite_target(
    df,
    component_cols,
    output_col,
    drop_components=True,
):
    """
    Create a composite target with priority:
    1 (any positive) > 0 (any zero) > -1 (all missing)

    Optionally drops component columns after creating the composite.
    """
    any_pos = (df[component_cols] == 1).any(axis=1)
    any_zero = (df[component_cols] == 0).any(axis=1)
    all_missing = (df[component_cols] == -1).all(axis=1)

    df[output_col] = np.select(
        [any_pos, any_zero, all_missing],
        [1, 0, -1],
        default=-1
    )

    # Drop component columns if requested
    if drop_components:
        df.drop(columns=component_cols, inplace=True, errors="ignore")

In [ ]:
def baseline_table_dev_test(
    dev_df: pd.DataFrame,
    test_df: pd.DataFrame,
    top_regimens: Optional[Sequence] = None,
    top_cancers: Optional[Sequence] = None,
    targets: Optional[Sequence] = None,
) -> pd.DataFrame:
    """
    Build a side-by-side baseline characteristics table (Dev vs Test).

    - Removed "Number of Treatments"
    - Added body_surface_area after height and weight
    - Added intent and cancer_type as count (%) for all levels (sorted by Dev)
    - Sex/Intent/Cancer type now also report Missing as a category row
    - Targets: count positives (==1); % is among assessable only (exclude -1 and NaN)
      Output format: "n_pos / n_assessable (pct%)"
    """

    if top_regimens is None:
        top_regimens = []
    if top_cancers is None:
        top_cancers = []
    if targets is None:
        targets = []

    def _fmt_count_pct(series: pd.Series, value, N: int) -> tuple[str, int]:
        """Return formatted 'n (pct%)' and the count for sorting. Handles Missing explicitly."""
        if value == "Missing":
            n = int(series.isna().sum())
        else:
            n = int((series == value).sum())
        pct = (n / N * 100) if N else 0.0
        return f"{n} ({pct:.1f}%)", n

    def _fmt_target_pos_assessable(
        series: pd.Series,
        pos_value: int = 1,
        missing_value: int = -1
    ) -> tuple[str, int]:
        """
        For targets coded as {-1, 0, 1} (or with NaNs):
        - assessable = values not in {missing_value, NaN}
        - n_pos = count(value == pos_value) among assessable
        - pct = n_pos / n_assessable
        Returns:
          display_str: "n_pos / n_assessable (pct%)"
          sort_n: n_pos (for sorting within characteristic)
        """
        s = pd.to_numeric(series, errors="coerce")
        assessable = s[(~s.isna()) & (s != missing_value)]
        denom = int(len(assessable))
        n_pos = int((assessable == pos_value).sum())
        pct = (n_pos / denom * 100) if denom else 0.0
        return f"{n_pos} / {denom} ({pct:.1f}%)", n_pos

    def _median_iqr(series: pd.Series, digits: int = 1) -> str:
        s = pd.to_numeric(series, errors="coerce")
        med = s.median()
        q25, q75 = s.quantile([0.25, 0.75])
        if pd.isna(med):
            return "NA"
        fmt = f"{{:.{digits}f}}"
        return f"{fmt.format(med)} ({fmt.format(q25)}–{fmt.format(q75)})"

    def _levels_with_missing(series: pd.Series) -> list:
        """
        Returns unique non-missing levels (as-is) plus a 'Missing' level if any NaNs exist.
        """
        levels = series.dropna().unique().tolist()
        if series.isna().any():
            levels.append("Missing")
        return levels

    def _rows_for_one(df: pd.DataFrame, cohort_name: str):
        N = len(df)
        rows = []

        # 1) Sex (counts/percent + Missing)
        if "sex" in df.columns:
            for lv in _levels_with_missing(df["sex"]):
                v, n = _fmt_count_pct(df["sex"], lv, N)
                rows.append(("Sex", str(lv), v, n))
        else:
            rows.append(("Sex", "NA", "NA", -1))

        # 2) Age
        rows.append(("Age (years)", "Median (IQR)", _median_iqr(df.get("age")), -1))

        # 3) Height
        rows.append(("Height (cm)", "Median (IQR)", _median_iqr(df.get("height")), -1))

        # 4) Weight
        rows.append(("Weight (kg)", "Median (IQR)", _median_iqr(df.get("weight")), -1))

        # 5) Body surface area
        rows.append(("Body surface area", "Median (IQR)", _median_iqr(df.get("body_surface_area")), -1))

        # 6) Intent (all levels + Missing)
        if "intent" in df.columns:
            for lv in _levels_with_missing(df["intent"]):
                v, n = _fmt_count_pct(df["intent"], lv, N)
                rows.append(("Intent", str(lv), v, n))
        else:
            rows.append(("Intent", "NA", "NA", -1))

        # 7) Cancer type (all levels + Missing)
        if "cancer_type" in df.columns:
            for lv in _levels_with_missing(df["cancer_type"]):
                v, n = _fmt_count_pct(df["cancer_type"], lv, N)
                rows.append(("Cancer type", str(lv), v, n))
        else:
            rows.append(("Cancer type", "NA", "NA", -1))

        # 8) Regimens (provided list)
        if top_regimens and "regimen" in df.columns:
            for reg in top_regimens:
                v, n = _fmt_count_pct(df["regimen"], reg, N)
                rows.append(("Regimen", str(reg), v, n))

        # 9) Cancer sites (provided list; optional separate block)
        if top_cancers and "cancer_type" in df.columns:
            for ca in top_cancers:
                v, n = _fmt_count_pct(df["cancer_type"], ca, N)
                rows.append(("Cancer site", str(ca), v, n))

        # 10) Targets (positives among assessable only)
        for t in targets:
            if t in df.columns:
                v, n = _fmt_target_pos_assessable(df[t], pos_value=1, missing_value=-1)
                rows.append((t.upper(), "No. (%)", v, n))
            else:
                rows.append((t.upper(), "No. (%)", "NA", -1))

        return pd.DataFrame(rows, columns=["Characteristic", "Category", cohort_name, "_sort_n"])

    dev = _rows_for_one(dev_df, "Dev")
    test = _rows_for_one(test_df, "Test")

    # Side-by-side merge, preserve Dev row order
    table = dev.merge(
        test.drop(columns=["_sort_n"]),
        on=["Characteristic", "Category"],
        how="outer",
        sort=False
    )

    # Preserve original row order
    table["_row_order"] = np.arange(len(table))

    # Sort ONLY within each characteristic by Dev count (descending), keeping block order
    table = table.sort_values(
        by=["Characteristic", "_sort_n", "_row_order"],
        ascending=[True, False, True],
        kind="mergesort"
    )

    # Restore characteristic block order exactly as it appears in dev-derived table
    char_first = table.groupby("Characteristic")["_row_order"].min()
    table["_char_order"] = table["Characteristic"].map(char_first)

    table = (
        table
        .sort_values(
            by=["_char_order", "_sort_n", "_row_order"],
            ascending=[True, False, True],
            kind="mergesort"
        )
        .drop(columns=["_sort_n", "_row_order", "_char_order"])
        .reset_index(drop=True)
    )

    return table

# Target Processing

## Remove unwanted targets

In [ ]:
# Suffixes of target columns to remove
suffixes_to_remove = ("_grade2plus", "_min", "_max", "_30d", "_60d", "_note")

# Collect target columns matching those suffixes and add specific targets to drop explicitly
targets_to_drop = (
    [c for c in main.columns if c.endswith(suffixes_to_remove)]
    + ["target_ED_CTAS_score", "target_H_length_of_stay", "target_ED2H"]
)

# Drop selected columns from the main df in place
main.drop(columns=targets_to_drop, inplace=True, errors="ignore")

## Create composite targets

In [ ]:
# Create a dictionary for composite targets
composite_targets = {
    "esas": {
        "component_cols": [
            c for c in main.columns
            if c.startswith("target_") and c.endswith("_3pt_change")
        ],
        "output_col": "target_any_esas_3pt_deterioration",
    },
    "hematological": {
        "component_cols": [
            "target_hemoglobin_grade3plus",
            "target_neutrophil_grade3plus",
            "target_platelet_grade3plus",
        ],
        "output_col": "target_any_hematological_grade3plus",
    },
    "hepatic": {
        "component_cols": [
            "target_bilirubin_grade3plus",
            "target_ALT_grade3plus",
            "target_AST_grade3plus",
        ],
        "output_col": "target_any_hepatic_grade3plus",
    },
}

In [ ]:
# Apply create_composite_target
for spec in composite_targets.values():
    create_composite_target(
        df=main,
        component_cols=spec["component_cols"],
        output_col=spec["output_col"],
        drop_components=True
    )

TODO: Check lookahead window for ESAS score deterioration

TODO: Add creatinine rise, which is a marker of AKI

## Add new targets (later)

TODO: Add targets from CT data

In [ ]:
# ct_mrns = set(ct["mrn"].dropna().unique())
# carg_mrns = set(carg["mrn"].dropna().unique())
# main_mrns = set(main["mrn"].dropna().unique())

# len(ct_mrns & main_mrns)

In [ ]:
# Add study-level start date (earliest AE date per study)
# ct['study_start_date'] = (
#     ct.groupby('study_name')['ae_grade_start_date']
#       .transform('min')
# )

# # Add study-level end date (latest AE date per study)
# ct['study_end_date'] = (
#     ct.groupby('study_name')['ae_grade_start_date']
#       .transform('max')
# )

In [ ]:
# ct[['study_name', 'study_start_date', 'study_end_date']].drop_duplicates()

In [ ]:
# ct['study_length_days'] = (
#     ct['study_end_date'] - ct['study_start_date']
# ).dt.days


In [ ]:
# import matplotlib.pyplot as plt
# study_lengths = (
#     ct[['study_name', 'study_length_days']]
#     .drop_duplicates()
# )

# plt.figure()
# plt.hist(study_lengths['study_length_days'].dropna(), bins=30)
# plt.xlabel('Study length (days)')
# plt.ylabel('Number of studies')
# plt.title('Distribution of Study Length (per study)')
# plt.show()


# Preprocessing

TODO: Find out what's wrong with line_of_therapy column

## Retrive first_treatment_date column

In [ ]:
# Add first treatment date from dates table
# Indices are the same
main["first_treatment_date"] = dates["first_treatment_date"]

In [ ]:
# Convert date columns to datetime and remove time component
date_cols = ["assessment_date", "first_treatment_date"]
main[date_cols] = main[date_cols].apply(
    lambda s: pd.to_datetime(s, errors="coerce").dt.floor("D")
)

## Create first_line_of_treatment column

TODO: For better accuracy, consider grouping by MRN + cancer type/site (not MRN alone), since a patient may develop multiple cancers over time and could have a distinct “first line of therapy” for each cancer.

In [ ]:
# Flag the first treatment line per patient
main["first_line_of_treatment"] = (
    main["first_treatment_date"]
    == main.groupby("mrn")["first_treatment_date"].transform("min")
)

## Value mapping

In [ ]:
# 1) preferred_language: replace with mapped_language when available
lang_map = (language_names_normalized
            .drop_duplicates("raw_language")
            .set_index("raw_language")["mapped_language"])

main["preferred_language"] = main["preferred_language"].map(lang_map).fillna(main["preferred_language"])

# Collapse rare language categories
s = main["preferred_language"]

# Categories with >=250 occurrences
keep = s.value_counts(dropna=False)
keep = keep[keep >= 250].index

# Replace rare categories with "Others" (keep missing as missing)
main["preferred_language"] = s.where(s.isna() | s.isin(keep), "Other")

In [ ]:
# 2) religion: use lev2 for Christian, otherwise lev1
rel_map = (
    religion_names_normalized
    .drop_duplicates("raw_religion")
    .assign(
        mapped_religion=lambda df: df["mapped_lev1_religion_name"].where(
            df["mapped_lev1_religion_name"] != "Christian",
            df["mapped_lev2_religion_name"]
        )
    )
    .set_index("raw_religion")["mapped_religion"]
)

main["religion"] = main["religion"].map(rel_map).fillna(main["religion"])

## Column Grouping

In [ ]:
meta_cols = ['mrn', 'assessment_date', 'first_treatment_date', 'primary_site_desc', 'study_drug', 'postalcode']
targ_cols = [col for col in main.columns if col.startswith('target') and col not in meta_cols]
feat_cols = main.columns.drop(meta_cols+targ_cols).tolist()

TODO: postalcode mapping

| FSA-based mapping              | How the mapping is done                     | Data source                                 | Why useful for toxicity prediction                                   |
| ------------------------------ | ------------------------------------------- | ------------------------------------------- | -------------------------------------------------------------------- |
| Urban vs rural                 | All 3 characters (FSA)                      | Statistics Canada (PCCF / census geography) | Proxy for access to oncology care, labs, and supportive services     |
| Income quintile (area-level)   | All 3 characters (FSA → area median income) | Statistics Canada Census                    | Captures socioeconomic status related to comorbidity and care access |
| Deprivation index (aggregated) | All 3 characters (FSA → CIMD domains)       | Statistics Canada (CIMD)                    | Reflects multidimensional social vulnerability                       |
| Province / health region       | First 1–2 characters                        | Canada Post / Statistics Canada             | Accounts for system-level differences in care delivery               |
| Remoteness / rurality proxy    | All 3 characters (FSA-based classification) | Statistics Canada (rurality / MIZ)          | Indicates travel burden and likelihood of ED use                     |


## Filter cohort

### Filter age

In [ ]:
# Filter to patients aged 65+
main_65 = main[main["age"] >= 65].copy()

# Record counts after age filter
n_rows_65 = len(main_65)
n_mrns_65 = main_65["mrn"].nunique()

### Filter first treatment(s)

In [ ]:
# NOTE: This would retain only the very first treatment per patient.
# For now, we keep all first_treatment_date records, so this is commented out.

# get first treatments only
# first_trt_idxs = dates.reset_index().groupby(['mrn','first_treatment_date']).first()['index'].tolist()
# main = main.loc[first_trt_idxs]

In [ ]:
# MRNs with at least one non-null first treatment date (before filtering)
mrns_with_any_first_treatment_date = set(
    main_65.loc[main_65["first_treatment_date"].notna(), "mrn"]
)

# Filter: only rows where assessment_date equals first_treatment_date
main_first_trts = main_65[
    main_65["first_treatment_date"].notna() &
    main_65["assessment_date"].notna() &
    (main_65["first_treatment_date"] == main_65["assessment_date"])
].copy()

# MRNs remaining after date-equality filter
mrns_with_equal_assessment_and_first_treatment_date = set(
    main_first_trts["mrn"]
)

# Counts AFTER the filter (patient-flow reporting)
n_rows_first_trts = len(main_first_trts)
n_mrns_first_trts = len(
    mrns_with_equal_assessment_and_first_treatment_date
)

# MRNs removed by the filter
mrns_removed_by_date_equality_filter = (
    mrns_with_any_first_treatment_date
    - mrns_with_equal_assessment_and_first_treatment_date
)

In [ ]:
# Prepare a summary
summary = {
    "n_mrns_with_any_first_treatment_date": len(mrns_with_any_first_treatment_date),
    "n_mrns_with_equal_dates": len(mrns_with_equal_assessment_and_first_treatment_date),
    "n_mrns_removed_by_date_equality_filter": len(mrns_removed_by_date_equality_filter),
    "percent_removed": (
        len(mrns_removed_by_date_equality_filter)
        / len(mrns_with_any_first_treatment_date)
    ) * 100,
}

summary

TODO: Some mrns_removed_by_date_equality_filter have a cycle 1 record, but the assessment_date doesn’t match the first_treatment_date. I think we should keep these cases, since the presence of cycle 1 should still represent the start of the regimen. For those MRNs, we may need to update first_treatment_date to the date of the cycle 1 start (i.e., the earliest treatment date in cycle 1), because I assume the treatment actually began when the first cycle started, not on the assessment date.

## Standardize regimens (later)

Check this: https://github.com/ml4oncology/make-clinical-dataset/blob/main/epic/scripts/ask_groq.py

# Data Integration

## Join EHR with OACC

In [ ]:
# Closest referral within 90 days BEFORE first treatment (per mrn x first_treatment_date row)
main_first_trts = merge_closest_measurements(
    main_first_trts,
    carg,
    main_date_col="first_treatment_date",
    meas_date_col="date_referred",
    direction="backward",
    time_window=(-90, 0),
    merge_individually=False,
    include_meas_date=True,   # keeps date_referred in output
)

In [ ]:
# Add more columns to meta_cols
meta_cols += ['date_referred', 'carg_toxicity_risk']

## Post-join correction

In [ ]:
# main_first_trts[main_first_trts["mrn"]==6207415][["regimen","first_treatment_date","first_line_of_treatment","date_referred","carg_toxicity_risk"]].head()

In [ ]:
# Ensure correct ordering
main_first_trts = (
    main_first_trts
    .sort_values(["mrn", "date_referred", "first_treatment_date"])
)

# Duplicate referral assignment mask
dup_mask = (
    main_first_trts["date_referred"].notna()
    &
    (
        main_first_trts
        .groupby(["mrn", "date_referred"])
        .cumcount()
        > 0
    )
)

# Null out duplicates
main_first_trts.loc[dup_mask, "date_referred"] = pd.NaT
main_first_trts.loc[dup_mask, "carg_toxicity_risk"] = np.nan

# Number of resolved issues
dup_mask.sum()

Because a wide lookahead window was used to link OACC referrals to lines of treatment, some referrals were initially assigned to more than one treatment line. We resolved **19** such cases by retaining each referral only for the line of treatment closest in time to the referral date.

**Example:** 

Assume **MRN = 12345** has **4 distinct treatment courses**, each with a different `treatment_start_date`.


**Treatment history**

| Treatment course | Treatment start date | Days since previous | CARG referral date | Matched by rule |
|------------------|---------------------|---------------------|--------------------|-----------------|
| Regimen 1        | Jan 1               | –                   | Feb 15             | ✅ (within 90 days) |
| Regimen 2        | Feb 20              | +50 days            | Feb 15             | ✅ (within 90 days) |
| Regimen 3        | May 10              | +79 days            | Feb 15             | ❌ (outside 90 days) |
| Regimen 4        | Aug 1               | +83 days            | Feb 15             | ❌ (outside 90 days) |



**Matching rule**

- A CARG referral is matched to a treatment course if it occurs **within 90 days before** the treatment start date.


**Observed behavior**

- The same CARG referral (**Feb 15**) falls within 90 days of:
  - **Regimen 1** (Jan 1)
  - **Regimen 2** (Feb 20)
- As a result, the referral is **matched to both treatment courses** under the current rule.


**Implication**

- After time-window matching:
  - Retain the CARG score **only for the earliest matched treatment per MRN**
  - Remove the CARG score from **subsequent treatments** for the same MRN

# Data Splitting

In [ ]:
# Set split date using the earliest assessment date with non-missing CARG
split_date = main_first_trts['date_referred'].min()
print(f'Temporal split at {split_date}')

In [ ]:
# Define mask for test set based on first treatment date
mask = main_first_trts["first_treatment_date"] >= split_date

# Apply split using mask
dev = main_first_trts[~mask].copy()
test = main_first_trts[mask].copy()

In [ ]:
# NOTE: Currently disabled.
# We keep all first_treatment_date records; enabling this would remove 879 sessions.

# (Prevents MRN leakage by excluding test MRNs from dev.)
# dev = dev[~dev["mrn"].isin(test["mrn"])]

In [ ]:
# Ensure no columns are left unclassified (meta / feature / target)
extra_cols = (
    set(main_first_trts.columns)
    - set(meta_cols)
    - set(feat_cols)
    - set(targ_cols)
)

assert not extra_cols, f"Unassigned columns found: {sorted(extra_cols)}"

In [ ]:
# Split dev data into metadata, targets, and features
dev_meta, dev_target, dev_feats = dev[meta_cols].copy(), dev[targ_cols].copy(), dev[feat_cols].copy()

# Split test data into metadata, targets, and features
test_meta, test_target, test_feats = test[meta_cols].copy(), test[targ_cols].copy(), test[feat_cols].copy()

# Label dataset split
dev_meta['split'] = 'Dev'
test_meta['split'] = 'Test'

# Combine dev and test for downstream processing
X, Y, meta = pd.concat([dev_feats, test_feats]), pd.concat([dev_target, test_target]), pd.concat([dev_meta, test_meta])

# Patient Flow

In [ ]:
# Dev counts
n_rows_dev = len(dev)
n_mrns_dev = dev["mrn"].nunique()

# Test counts
n_rows_test = len(test)
n_mrns_test = test["mrn"].nunique()

# Test rows/MRNs with available CARG (no subsetting created)
carg_mask = test["carg_toxicity_risk"].notna()

n_rows_test_carg = carg_mask.sum()
n_mrns_test_carg = test.loc[carg_mask, "mrn"].nunique()

In [ ]:
patient_flow_table = pd.DataFrame(
    [
        ("All data", n_rows_all, n_mrns_all),
        ("Age ≥ 65", n_rows_65, n_mrns_65),
        ("First treatment only", n_rows_first_trts, n_mrns_first_trts),
        ("Development set", n_rows_dev, n_mrns_dev),
        ("Test set", n_rows_test, n_mrns_test),
        ("Test set with CARG score", n_rows_test_carg, n_mrns_test_carg),
    ],
    columns=["Step", "Rows", "Unique MRNs"]
)

patient_flow_table

TODO: Confirm whether having substantially more records in dev than in test is okay.

TODO: Find MRNs with no chemo info

# Label distribution

TODO: Decide on an approach to address class imbalance (e.g., class weighting, resampling, threshold tuning, or imbalance-aware metrics).

In [ ]:
# Per session
get_label_distribution(Y, meta, with_respect_to='sessions')

In [ ]:
# Per patient
get_label_distribution(Y, meta, with_respect_to='patients')

# Baseline Charachteristics

TODO: Define a function to report characteristics for a selected dataset (overall, development, or test).

Side by side, median IQR, TARGET RATE

In [ ]:
main_first_trts["sex"].value_counts()

In [ ]:
test["sex"].isna().sum()


In [ ]:
baseline_table = baseline_table_dev_test(dev_df=dev, test_df=test,targets=targ_cols)
baseline_table

# Model Development

In [ ]:
# Date-stamped base directory
date_str = datetime.today().strftime("%Y%m%d")
base_dir = f"{WORKING_DIR}/AutogluonModels/{date_str}"

In [ ]:
# Train models (saved to base_dir/<target>/)
_ = train_models(
    dev_feats, 
    dev_target,
    dev_meta, 
    save_path=base_dir
)

In [ ]:
base_dir = f"{WORKING_DIR}/AutogluonModels/20260119"

In [ ]:
# Load trained models
models = {}

for target in dev_target.columns:
    model_dir = f"{base_dir}/{target}"
    if os.path.exists(model_dir):
        models[target] = TabularPredictor.load(model_dir, verbosity=0)

# Model Evaluation

In [ ]:
# Collect validation performance
res = {}
for target in models:
    res[target] = models[target].leaderboard()[['model', 'score_val']]

pd.concat(res, axis=1)

In [ ]:
go_treat_eval = evaluate(models, test_feats, test_target)
go_treat_eval

In [ ]:
# Transpose evaluation output so that (target, metric) move from columns to the index
go_treat_eval_T = go_treat_eval.T

# Drop the dummy column dimension (each metric has a single value per target)
go_treat_eval_T = go_treat_eval_T.iloc[:, 0]

# Reshape so that each target is a row and metrics (model, roc_auc, average_precision) are columns
go_treat_long = go_treat_eval_T.unstack(level=1)

# Extract AUROC only and rename for clarity in comparison tables
go_treat_auc = (
    go_treat_long[["roc_auc"]]
    .rename(columns={"roc_auc": "GO_TREAT_AUROC"})
)

# Explicitly name the index to ensure alignment when joining with CARG results
go_treat_auc.index.name = "target"

# Sanity check: confirm targets are correctly set as the index
print(go_treat_long.index)

TODO: Handle this:
LightGBM failed during training due to a feature-alignment mismatch caused by the custom data splitting logic in the training function. Specifically, the function manually separates training and tuning data after feature generation, which can lead to slight differences in the processed feature sets (e.g., text-derived or binned features) between splits. LightGBM enforces strict feature consistency across datasets and therefore raised an error when expected features were missing, whereas other models were able to proceed despite this mismatch.

# Evaluation of CARG Risk Score Discrimination

In [ ]:
# Keep patients with CARG score
mask = test_meta["carg_toxicity_risk"].notna()

# Encode CARG risk as ordinal score
carg_score = (
    test_meta.loc[mask, "carg_toxicity_risk"]
    .map({"Low": 1, "Moderate": 2, "High": 3})
    .astype(int)
)

# Compute AUROC per target
carg_res = {}
for target, label in test_target[mask].items():
    valid = label != -1
    carg_res[target] = auc_scores(
        label[valid],
        carg_score[valid]
    )["AUROC"]   # explicitly take AUROC only

# Convert to DataFrame
carg_auc = pd.DataFrame.from_dict(
    carg_res,
    orient="index",
    columns=["CARG_AUROC"]
)

carg_auc.index.name = "target"

# Comparative Analysis

In [ ]:
# Side-by-side AUROC comparison
auroc_comparison = (
    go_treat_auc
    .join(carg_auc)
)

# Delta AUROC (GO-TREAT – CARG)
auroc_comparison["delta_AUROC"] = (
    auroc_comparison["GO_TREAT_AUROC"]
    - auroc_comparison["CARG_AUROC"]
)

# Round for reporting
auroc_comparison = (
    auroc_comparison[["CARG_AUROC", "GO_TREAT_AUROC", "delta_AUROC"]]
    .astype(float)
    .round(4)
    .sort_values("delta_AUROC", ascending=False)
)
auroc_comparison

Interpretation: 
The table shows that **GO-TREAT consistently outperforms the CARG score** in terms of discrimination (AUROC) across all evaluated outcomes. The largest absolute improvement is observed for **severe hepatic toxicity**, where CARG shows almost no discriminative ability (AUROC ≈ 0.05), while GO-TREAT achieves moderate discrimination (AUROC ≈ 0.66). Substantial gains are also seen for **AKI grade ≥3** (ΔAUROC ≈ 0.25) and **hospitalization within 90 days** (ΔAUROC ≈ 0.20), indicating that the machine learning model captures clinically relevant risk factors not reflected in the CARG score.

Moderate but consistent improvements are observed for **hematological toxicity**, **symptom deterioration**, and **1-year mortality**, with ΔAUROC values ranging from approximately 0.15 to 0.17. Even for outcomes where CARG already demonstrates reasonable discrimination, such as **mortality** and **ED visits**, GO-TREAT provides additional improvement. Overall, these results suggest that integrating routinely collected EHR data through GO-TREAT substantially enhances risk stratification beyond the traditional CARG score, particularly for outcomes not explicitly targeted by CARG.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

bars = auroc_comparison["delta_AUROC"].plot(
    kind="barh",
    ax=ax
)

ax.axvline(0, linestyle="--")
ax.set_xlabel("Δ AUROC (GO-TREAT − CARG)")
ax.set_ylabel("Outcome")
ax.set_title("Improvement in Discrimination (AUROC): GO-TREAT vs CARG")

# Add value labels
for i, v in enumerate(auroc_comparison["delta_AUROC"]):
    ax.text(
        v + 0.01,           # shift slightly right
        i,
        f"{v:.3f}",
        va="center"
    )

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

y = range(len(auroc_comparison))

ax.hlines(
    y=y,
    xmin=auroc_comparison["CARG_AUROC"],
    xmax=auroc_comparison["GO_TREAT_AUROC"]
)

ax.plot(auroc_comparison["CARG_AUROC"], y, marker="o", linestyle="", label="CARG")
ax.plot(auroc_comparison["GO_TREAT_AUROC"], y, marker="o", linestyle="", label="GO-TREAT")

# Add labels
for i, row in enumerate(auroc_comparison.itertuples()):
    ax.text(row.CARG_AUROC - 0.02, i, f"{row.CARG_AUROC:.3f}", va="center", ha="right")
    ax.text(row.GO_TREAT_AUROC + 0.01, i, f"{row.GO_TREAT_AUROC:.3f}", va="center", ha="left")

ax.set_yticks(y)
ax.set_yticklabels(auroc_comparison.index)
ax.set_xlabel("AUROC")
ax.set_title("AUROC Comparison: CARG vs GO-TREAT")
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

auroc_comparison[["CARG_AUROC", "GO_TREAT_AUROC"]].plot(
    kind="bar",
    ax=ax
)

ax.set_ylabel("AUROC")
ax.set_title("AUROC Comparison by Outcome")
ax.set_xticklabels(auroc_comparison.index, rotation=45, ha="right")

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3)

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------
# Top-10 most important features per target (best model)
# ------------------------------------------------------------
feature_importance_by_target = {}

for target, predictor in models.items():
    print(f"\n=== Feature importance for target: {target} ===")

    # Best model (leaderboard is always sorted best -> worst)
    best_model = predictor.leaderboard().iloc[0]["model"]

    # Build dataset required for feature importance
    fi_data = test_feats.copy()
    fi_data[target] = test_target[target]

    # Remove censored / invalid labels (consistent with training)
    mask = test_target[target] != -1
    fi_data = fi_data.loc[mask]

    # Compute permutation importance
    fi = predictor.feature_importance(
        data=fi_data,
        model=best_model,
        subsample_size=5000  # reduce if slow
    )

    top10 = (
        fi
        .reset_index()
        .rename(columns={"index": "feature"})
        .sort_values("importance", ascending=False)
        .head(10)
    )

    feature_importance_by_target[target] = top10
    display(top10)
